# Requirement

### I have a BRD (Business Requirment Document), <br> 1. I want to prepare a Data Engineering solution Document including Requirements <br> 2. Python Code for its solution <br> Finally Execute a code to achieve final goal accroding to the BRD<br> BRD Agent AI : To Analyze the BRD Requirements. <br> Schema Agent : To Analyze the Schema and Write Qureies based on Input <br> Solution Agent : To analyze BRD and Schema and Generate Solution

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

if os.environ.get("OPENAI_API_KEY"):
    print("OPENAI_API_KEY is available.")
else:
    print("OPENAI_API_KEY is not available.") 

if os.environ.get("ORACLEDSN"):
    print("ORACLEDSN is available.")
else:
    print("ORACLEDSN is not available.") 


if os.environ.get("ORACLEUSERNAME"):
    print("ORACLEUSERNAME is available.")
else:
    print("ORACLEUSERNAME is not available.") 


if os.environ.get("ORACLEPWD"):
    print("ORACLEPWD is available.")
else:
    print("ORACLEPWD is not available.")

OPENAI_API_KEY is available.
ORACLEDSN is available.
ORACLEUSERNAME is available.
ORACLEPWD is available.


### Extracting Text from ADM Customer Schema to Undestand By Agent

In [2]:
from langchain_community.document_loaders import PyPDFLoader
schemaloader=PyPDFLoader("./Source/ADM_Customer_Orders_Schema.pdf")
schemadoc=schemaloader.load()

C:\Users\navat\AppData\Local\Temp\ipykernel_17668\4207395807.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [3]:
schemadoc

[Document(metadata={'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2026-07-06T16:29:46+05:30', 'author': 'Naresh Neelam', 'moddate': '2026-07-06T16:29:46+05:30', 'source': './Source/ADM_Customer_Orders_Schema.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1'}, page_content="ADM Customer Orders Sample Schema  \nSchema Description  \nCustomer Orders (CO) is a Oracle schema resembling a generic customer orders management schema. The Customer \nOrders (CO) schema records the details of transactions made by a retail application.  \nThe company sells a variety of products, which are maintained in the products table. Each product has a unique \nidentification number, name, price, details stored in a JSON object and product image details.  \nThe orders placed by the customer are tracked in the orders table using the order identification number, date and \ntime when the order was placed, customer details, order status and the store information.  \nThe

### Create Own Metadata for ADM Customer Schema PDF Chunks

In [4]:
for i in schemadoc:
    i.metadata = {
        "Source": "ADM_Customer_Orders",
        "ID": "Schema",
        "Creation_Date": "2026-07-06T18:55:24+05:30",
        "Author": "Naresh Neelam",
        "JIRA": "SCH0001"
    }

In [5]:
schemadoc

[Document(metadata={'Source': 'ADM_Customer_Orders', 'ID': 'Schema', 'Creation_Date': '2026-07-06T18:55:24+05:30', 'Author': 'Naresh Neelam', 'JIRA': 'SCH0001'}, page_content="ADM Customer Orders Sample Schema  \nSchema Description  \nCustomer Orders (CO) is a Oracle schema resembling a generic customer orders management schema. The Customer \nOrders (CO) schema records the details of transactions made by a retail application.  \nThe company sells a variety of products, which are maintained in the products table. Each product has a unique \nidentification number, name, price, details stored in a JSON object and product image details.  \nThe orders placed by the customer are tracked in the orders table using the order identification number, date and \ntime when the order was placed, customer details, order status and the store information.  \nThe details of the products in a particular order are also tracked in the adm_order_items table using the order \nidentification number. Details o

### Splitting ADM Customer Schema Documents into Chunks

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

schmatextsplitter=RecursiveCharacterTextSplitter(
      chunk_size=1000,
      chunk_overlap=100
)

schemadocchunks=schmatextsplitter.split_documents(schemadoc)

In [7]:
schemadocchunks

[Document(metadata={'Source': 'ADM_Customer_Orders', 'ID': 'Schema', 'Creation_Date': '2026-07-06T18:55:24+05:30', 'Author': 'Naresh Neelam', 'JIRA': 'SCH0001'}, page_content='ADM Customer Orders Sample Schema  \nSchema Description  \nCustomer Orders (CO) is a Oracle schema resembling a generic customer orders management schema. The Customer \nOrders (CO) schema records the details of transactions made by a retail application.  \nThe company sells a variety of products, which are maintained in the products table. Each product has a unique \nidentification number, name, price, details stored in a JSON object and product image details.  \nThe orders placed by the customer are tracked in the orders table using the order identification number, date and \ntime when the order was placed, customer details, order status and the store information.  \nThe details of the products in a particular order are also tracked in the adm_order_items table using the order \nidentification number. Details o

### Embeddings Preparation for ADM Customer Schema

In [8]:
from langchain_openai import OpenAIEmbeddings
embeddingmodel=OpenAIEmbeddings(model="text-embedding-3-small")

### Store Embeddings in Vector DB for ADM Customer Schema

In [9]:
from langchain_community.vectorstores import Chroma
VectorDB=Chroma.from_documents(
    documents=schemadocchunks,
    embedding=embeddingmodel
)

### Semantic Search for ADM Customer Schema

In [10]:
VectorDB.similarity_search("*")

[Document(metadata={'JIRA': 'SCH0001', 'ID': 'Schema', 'Author': 'Naresh Neelam', 'Creation_Date': '2026-07-06T18:55:24+05:30', 'Source': 'ADM_Customer_Orders'}, page_content='inventory_pk, inventory_store_product_u, shipments_store_id_i, \nshipments_customer_id_i, shipments_pk  \nTable  adm_customers, adm_stores, adm_products, adm_orders, \nadm_order_items, adm_shipments, adm_inventory  \nView  customer_order_products, store_orders, product_reviews, product_orders  \n \n Table Descriptions  \nADM_CUSTOMERS \nName Null Type Relationships \nCUSTOMER_ID NOT \nNULL INTEGER relates each row of the table customers to none or more \nrows of customer_id in the orders table. \nEMAIL_ADDRESS NOT \nNULL VARCHAR2(255)'),
 Document(metadata={'JIRA': 'SCH0001', 'ID': 'Schema', 'Creation_Date': '2026-07-06T18:55:24+05:30', 'Source': 'ADM_Customer_Orders', 'Author': 'Naresh Neelam'}, page_content='Name Null Type Relationships \nFULL_NAME NOT \nNULL VARCHAR2(255)  \n \nADM_STORES \nName Null Type Rela

In [11]:
schemacontext = VectorDB.similarity_search(
       "*",
       filter={"$and": [{"Source": "ADM_Customer_Orders"}, {"ID": "Schema"}]}
)

### Extracting Text from ADM Cutomers BRD to Undestand By BRD Agent

In [12]:
from langchain_community.document_loaders import PyPDFLoader
schemaloader=PyPDFLoader("./Source/ADM_Customer_Orders_BRD.pdf")
brddoc=schemaloader.load()

In [13]:
brddoc

[Document(metadata={'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2026-07-06T12:30:14+05:30', 'author': 'Naresh Neelam', 'moddate': '2026-07-06T12:30:14+05:30', 'source': './Source/ADM_Customer_Orders_BRD.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}, page_content='REQUIREMNT \nNUMBER \nREQUIREMENT TYPE \nDFUN0001 Identify High-Value "Loyal" Customers \n \nFind customers who have placed at least 5 orders and whose \ntotal spending exceeds the average spending of all \ncustomers in the database. \nFunctional \nDFUN0002 Find the Top-Selling Product per Category \n \nRetrieve the single top-selling product (by cumulative sales \namount) within each individual product category \nFunctional \nDFUN0003 Detect Dormant Accounts (Zero Purchases) \n \nGenerate a list of all registered customers who have never \nplaced an order, or haven\'t ordered anything within the last \ncalendar year. \nFunctional \nDFUN0004 Pinpoint Consecutive Sales Surges 

### Create Own Metadata for ADM Customer BRD PDF Chunks

In [15]:
for i in brddoc:
    i.metadata = {
        "Source": "ADM_Customer_Orders_brd",
        "ID": "BRD",
        "Creation_Date": "2026-07-06T18:55:24+05:30",
        "Author": "Naresh Neelam",
        "JIRA": "BRD0001"
    }

In [16]:
brddoc

[Document(metadata={'Source': 'ADM_Customer_Orders_brd', 'ID': 'BRD', 'Creation_Date': '2026-07-06T18:55:24+05:30', 'Author': 'Naresh Neelam', 'JIRA': 'BRD0001'}, page_content='REQUIREMNT \nNUMBER \nREQUIREMENT TYPE \nDFUN0001 Identify High-Value "Loyal" Customers \n \nFind customers who have placed at least 5 orders and whose \ntotal spending exceeds the average spending of all \ncustomers in the database. \nFunctional \nDFUN0002 Find the Top-Selling Product per Category \n \nRetrieve the single top-selling product (by cumulative sales \namount) within each individual product category \nFunctional \nDFUN0003 Detect Dormant Accounts (Zero Purchases) \n \nGenerate a list of all registered customers who have never \nplaced an order, or haven\'t ordered anything within the last \ncalendar year. \nFunctional \nDFUN0004 Pinpoint Consecutive Sales Surges \n \nWrite a query to find all instances where an individual \ncustomer\'s transaction amount exceeded $1,000 for three \nconsecutive calen

### Splitting ADM Customer BRD Documents into Chunks

In [18]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

brdtextsplitter=RecursiveCharacterTextSplitter(
      chunk_size=1000,
      chunk_overlap=100
)

brddocchunks=brdtextsplitter.split_documents(brddoc)

In [19]:
brddocchunks

[Document(metadata={'Source': 'ADM_Customer_Orders_brd', 'ID': 'BRD', 'Creation_Date': '2026-07-06T18:55:24+05:30', 'Author': 'Naresh Neelam', 'JIRA': 'BRD0001'}, page_content='REQUIREMNT \nNUMBER \nREQUIREMENT TYPE \nDFUN0001 Identify High-Value "Loyal" Customers \n \nFind customers who have placed at least 5 orders and whose \ntotal spending exceeds the average spending of all \ncustomers in the database. \nFunctional \nDFUN0002 Find the Top-Selling Product per Category \n \nRetrieve the single top-selling product (by cumulative sales \namount) within each individual product category \nFunctional \nDFUN0003 Detect Dormant Accounts (Zero Purchases) \n \nGenerate a list of all registered customers who have never \nplaced an order, or haven\'t ordered anything within the last \ncalendar year. \nFunctional \nDFUN0004 Pinpoint Consecutive Sales Surges \n \nWrite a query to find all instances where an individual \ncustomer\'s transaction amount exceeded $1,000 for three \nconsecutive calen

### Embeddings and Storage in Vecord DB for ADM Customer BRD Doc

In [20]:
from langchain_openai import OpenAIEmbeddings
embeddingmodel=OpenAIEmbeddings(model="text-embedding-3-small")

from langchain_community.vectorstores import Chroma
VectorDB=Chroma.from_documents(
    documents=brddocchunks,
    embedding=embeddingmodel
)


### Semantic Search for ADM Customer BRD and Create Context

In [21]:
brdcontext = VectorDB.similarity_search(
       "*",
       filter={"$and": [{"Source": "ADM_Customer_Orders_brd"}, {"ID": "BRD"}]}
)

In [22]:
brdcontext

[Document(metadata={'JIRA': 'BRD0001', 'ID': 'BRD', 'Source': 'ADM_Customer_Orders_brd', 'Author': 'Naresh Neelam', 'Creation_Date': '2026-07-06T18:55:24+05:30'}, page_content="customer's timeline to evaluate moving spending habits \nFunctional \nNFUN0001 All the Above details send to Email Notification with each \nheading in a single email with Subject : ADM Customers \nReports \n \nEmail id: naresh_neelam@outlook.com \nTechnical \n                           ADM CUSTOMER DATA – REPORTING BRD"),
 Document(metadata={'Source': 'ADM_Customer_Orders_brd', 'ID': 'BRD', 'JIRA': 'BRD0001', 'Author': 'Naresh Neelam', 'Creation_Date': '2026-07-06T18:55:24+05:30'}, page_content='REQUIREMNT \nNUMBER \nREQUIREMENT TYPE \nDFUN0001 Identify High-Value "Loyal" Customers \n \nFind customers who have placed at least 5 orders and whose \ntotal spending exceeds the average spending of all \ncustomers in the database. \nFunctional \nDFUN0002 Find the Top-Selling Product per Category \n \nRetrieve the sing

### Agents Creation

In [38]:
from agents import Agent, Runner
from langchain_openai import ChatOpenAI


#Schema Agent
schmaagent = Agent(
    name="Schema Agent",
    instructions=f"You are an expert assistant to analyze the Database Schema and its tables and Generate Oracle SQL Query based on Request.\
                  Use ONLY the following retrived context to answer: --CONTEXT--- {schemacontext}---END CONTEXT---",
    model="gpt-5-nano"
)


# BRD Agent
brdagent = Agent(
    name="BRD Agent",
    instructions=f"You are an expert assistant to analyze the Business Requirements.\
                  Use ONLY the following retrived context to answer: --CONTEXT--- { brdcontext }---END CONTEXT---",
     model="gpt-5-nano"
)


#Solution Agent

solutionAgent = Agent(
    name="Solution Agent",
    instructions=f"You are an expert assistant to analyze the Business Requirements with Schema map\
                    Use ONLY the following retrived context to answer: --CONTEXT--- { brdcontext, schemacontext }---END CONTEXT---",
     model="gpt-5-nano"
)


### Handsoff Agents - 

In [42]:
from agents import Agent, Runner
# Step 2: Define the triage/router agent and attach handoffs

triage_agent = Agent(
    name="Triage Agent",
    instructions=(
        "Determine the intension of the user's request. "
        "If the request is related SQL Queries Generation or Schema, hand off the conversion to the Schema Agent "
        "If the request is related BRD Analysis , hand off the conversion to the BRD Agent."
        "If the request is related BRD and schema both, hand off the conversion to the Solution Agent."
    ),
    # Passing other agents in the handoffs list exposes them as available routes
    handoffs=[schmaagent, brdagent, solutionAgent],
)

### Running Agent - Schema Agent

In [35]:
from agents import Agent, Runner

# 3. Execute the multi-agent system using a Runner
runner = Runner()
result = runner.run(
    triage_agent,
    "Hi, how many columns are available in the Table ADM_CUSTOMERS"
)

# await the coroutine to get the actual result object, then print agent name
result = await result
print(f"Agent Used:"+ result.last_agent.name)
print(f"Agent Output:"+ result.final_output)

Agent Used:Schema Agent
Agent Output:ADM_CUSTOMERS has 3 columns: CUSTOMER_ID, EMAIL_ADDRESS, and FULL_NAME.


### Running Agent - BRD Agent

In [37]:
# 3. Execute the multi-agent system using a Runner
runner = Runner()
result = runner.run(
    triage_agent,
    "Hi, please analyse the BRD and provide the solution for each requirement"
)

# await the coroutine to get the actual result object, then print agent name
result = await result
print(f"Agent Used:"+ result.last_agent.name)
print(f"Agent Output:"+ result.final_output)

Agent Used:BRD Agent
Agent Output:Here are solutions aligned to each BRD requirement (DFUN0001–DFUN0005) using the data implied in the BRD. Assumes standard tables: orders(o), order_items(oi), products(p), categories(c), customers(cust). Adjust names as needed.

1) DFUN0001 Identify High-Value "Loyal" Customers
- Objective: Customers with at least 5 orders and total_spend > average total_spend across all customers.
- Data sources: orders (customer_id, order_id, order_date, total_amount)
- Approach: compute per-customer order_count and total_spent; compare to overall average total_spent.
- SQL (example):
  WITH customer_spending AS (
    SELECT o.customer_id,
           COUNT(*) AS order_count,
           SUM(o.total_amount) AS total_spent
    FROM orders o
    GROUP BY o.customer_id
  ),
  avg_spent AS (
    SELECT AVG(total_spent) AS avg_total_spent FROM customer_spending
  )
  SELECT cs.customer_id, cs.order_count, cs.total_spent
  FROM customer_spending cs CROSS JOIN avg_spent a
  W

### Running Agent - Solution Agent

In [45]:
runner = Runner()
result = runner.run(
    triage_agent,
    "Hi, please analyse bothe BRD and Schema and provide the solution with tables mentioned in schema for each requirement \
    ( functional and Technical email notification as well) in python\
     for Oracle Connection : use env variables -  ORACLEUSERNAME for user name, ORACLEPWD for pwd, ORACLEDSN for oracle DSN \
     save the output to ./Ouput/ "
)

# await the coroutine to get the actual result object, then print agent name
result = await result
print(f"Agent Used:"+ result.last_agent.name)
print(f"Agent Output:"+ result.final_output)

Agent Used:Solution Agent
Agent Output:Here's a self-contained Python solution that maps the BRD and Schema requirements to SQL queries against the Oracle schema, and saves outputs to ./Ouput/. It uses environment variables for the Oracle connection and includes functional and basic technical email scaffolding.

What this does
- Maps each BRD requirement (DFUN0001–DFUN0005) to a query against the provided ADM_* schema and the product_orders view where applicable.
- Creates output CSVs for each requirement under ./Ouput/.
- Builds a consolidated functional email body (one email with all headings).
- Includes a basic email sending scaffold (optional, uses SMTP env vars if provided).

Note: The Top-Selling Product per Category assumes a category field available via the product_orders view or a similar source (as the provided ADM_PRODUCTS table schema does not contain a category column). If your environment uses an explicit CATEGORY column in a view/table, the query is already structured t

#### I have copied the python code and saved in Output directory